In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim


# ============================================================
# 1. TRAFFIC ENVIRONMENT
# ============================================================

class TrafficEnvironment:

    def __init__(self, num_agents=4):
        self.num_agents = num_agents
        self.state_size = 3
        self.action_size = 2

        # State:
        # [queue_length, waiting_time, traffic_density]
        self.state = np.zeros(
            (num_agents, self.state_size)
        )

    def reset(self):

        self.state = np.random.randint(
            0, 10,
            (self.num_agents, self.state_size)
        ).astype(float)

        return self.state

    def predict_state(self, state, action):

        future_state = state.copy()

        if action == 1:
            # Green signal
            future_state[0] = max(
                0, future_state[0] - 2
            )

            future_state[1] = max(
                0, future_state[1] - 2
            )

        else:
            # Red signal
            future_state[0] += 1
            future_state[1] += 1

        return future_state

    def step(self, actions):

        rewards = []

        for i in range(self.num_agents):

            queue = self.state[i][0]
            waiting = self.state[i][1]
            density = self.state[i][2]

            if actions[i] == 1:

                queue = max(
                    0, queue - 2
                )

                waiting = max(
                    0, waiting - 2
                )

            else:

                queue += 1
                waiting += 1

            density += np.random.randint(
                -1, 2
            )

            density = np.clip(
                density, 0, 10
            )

            self.state[i] = [
                queue,
                waiting,
                density
            ]

            # Reward function
            reward = -(
                queue +
                waiting +
                density
            )

            rewards.append(reward)

        return self.state, rewards


# ============================================================
# 2. ACTOR NETWORK
# ============================================================

class Actor(nn.Module):

    def __init__(
        self,
        state_size,
        action_size
    ):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                state_size,
                64
            ),

            nn.ReLU(),

            nn.Linear(
                64,
                64
            ),

            nn.ReLU(),

            nn.Linear(
                64,
                action_size
            ),

            nn.Softmax(dim=-1)
        )

    def forward(self, state):

        return self.network(state)


# ============================================================
# 3. CRITIC NETWORK
# ============================================================

class Critic(nn.Module):

    def __init__(
        self,
        total_state_size,
        total_action_size
    ):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                total_state_size +
                total_action_size,
                128
            ),

            nn.ReLU(),

            nn.Linear(
                128,
                64
            ),

            nn.ReLU(),

            nn.Linear(
                64,
                1
            )
        )

    def forward(
        self,
        states,
        actions
    ):

        x = torch.cat(
            [states, actions],
            dim=1
        )

        return self.network(x)


# ============================================================
# 4. TRAFFIC AGENT
# ============================================================

class TrafficAgent:

    def __init__(
        self,
        state_size,
        action_size
    ):

        self.actor = Actor(
            state_size,
            action_size
        )

        self.actor_optimizer = optim.Adam(
            self.actor.parameters(),
            lr=0.001
        )

    def select_action(
        self,
        state
    ):

        state_tensor = torch.FloatTensor(
            state
        ).unsqueeze(0)

        probabilities = self.actor(
            state_tensor
        )

        action = torch.argmax(
            probabilities
        ).item()

        return action

    def adapt(
        self,
        reward
    ):

        # Basic meta-adaptation
        # Positive reward indicates
        # better traffic control.

        if reward > -10:

            learning_rate = 0.0001

        else:

            learning_rate = 0.00005

        for parameter in self.actor.parameters():

            parameter.data += (
                learning_rate *
                torch.randn_like(parameter)
            )


# ============================================================
# 5. MPC PLANNER
# ============================================================

class MPCPlanner:

    def __init__(
        self,
        environment
    ):

        self.environment = environment

    def calculate_cost(
        self,
        state
    ):

        queue = state[0]
        waiting = state[1]
        density = state[2]

        return (
            queue +
            waiting +
            density
        )

    def select_action(
        self,
        state
    ):

        # Predict state for action 0
        future_red = (
            self.environment.predict_state(
                state,
                0
            )
        )

        # Predict state for action 1
        future_green = (
            self.environment.predict_state(
                state,
                1
            )
        )

        cost_red = self.calculate_cost(
            future_red
        )

        cost_green = self.calculate_cost(
            future_green
        )

        if cost_green < cost_red:

            return 1

        return 0


# ============================================================
# 6. HIGH-LEVEL CONTROLLER
# ============================================================

class HierarchicalController:

    def __init__(
        self,
        num_agents
    ):

        self.num_agents = num_agents

    def get_goal(
        self,
        states
    ):

        total_queue = np.sum(
            states[:, 0]
        )

        total_waiting = np.sum(
            states[:, 1]
        )

        if total_queue > 20:

            return "REDUCE_CONGESTION"

        if total_waiting > 20:

            return "REDUCE_WAITING"

        return "NORMAL_TRAFFIC"


# ============================================================
# 7. MULTI-AGENT TRAFFIC SYSTEM
# ============================================================

class TrafficOptimizationSystem:

    def __init__(
        self,
        num_agents=4
    ):

        self.num_agents = num_agents

        self.environment = (
            TrafficEnvironment(
                num_agents
            )
        )

        self.agents = [

            TrafficAgent(
                state_size=3,
                action_size=2
            )

            for _ in range(
                num_agents
            )
        ]

        self.mpc = MPCPlanner(
            self.environment
        )

        self.controller = (
            HierarchicalController(
                num_agents
            )
        )

    def run_episode(
        self,
        steps=20
    ):

        states = (
            self.environment.reset()
        )

        total_reward = 0

        goal = (
            self.controller.get_goal(
                states
            )
        )

        print(
            "High-Level Goal:",
            goal
        )

        for step in range(steps):

            actions = []

            for i in range(
                self.num_agents
            ):

                # MADDPG-style policy action
                policy_action = (
                    self.agents[i]
                    .select_action(
                        states[i]
                    )
                )

                # MPC planning action
                mpc_action = (
                    self.mpc.select_action(
                        states[i]
                    )
                )

                # If traffic is high,
                # prefer MPC decision.
                if (
                    states[i][0] +
                    states[i][1]
                    > 10
                ):

                    final_action = (
                        mpc_action
                    )

                else:

                    final_action = (
                        policy_action
                    )

                actions.append(
                    final_action
                )

            next_states, rewards = (
                self.environment.step(
                    actions
                )
            )

            for i in range(
                self.num_agents
            ):

                self.agents[i].adapt(
                    rewards[i]
                )

            states = next_states

            total_reward += sum(
                rewards
            )

        return total_reward


# ============================================================
# 8. TRAINING
# ============================================================

def main():

    print("=" * 50)

    print(
        "MULTI-AGENT TRAFFIC OPTIMIZATION"
    )

    print("=" * 50)

    print(
        "Algorithm: MADDPG + MPC"
    )

    print(
        "Hierarchical Learning: Enabled"
    )

    print(
        "Meta-Learning: Enabled"
    )

    print("=" * 50)

    system = (
        TrafficOptimizationSystem(
            num_agents=4
        )
    )

    episodes = 10

    rewards_history = []

    for episode in range(
        episodes
    ):

        total_reward = (
            system.run_episode(
                steps=20
            )
        )

        rewards_history.append(
            total_reward
        )

        print(
            "Episode:",
            episode + 1,
            "| Total Reward:",
            round(
                total_reward,
                2
            )
        )

        print("-" * 50)

    print(
        "\nTraining Completed!"
    )

    print(
        "Average Reward:",
        round(
            np.mean(
                rewards_history
            ),
            2
        )
    )

    print(
        "Best Reward:",
        round(
            max(rewards_history),
            2
        )
    )

    print(
        "Worst Reward:",
        round(
            min(rewards_history),
            2
        )
    )

    print(
        "\nTraffic optimization completed."
    )


# ============================================================
# 9. RUN PROGRAM
# ============================================================

if __name__ == "__main__":

    main()

MULTI-AGENT TRAFFIC OPTIMIZATION
Algorithm: MADDPG + MPC
Hierarchical Learning: Enabled
Meta-Learning: Enabled
High-Level Goal: REDUCE_WAITING
Episode: 1 | Total Reward: -969.0
--------------------------------------------------
High-Level Goal: REDUCE_WAITING
Episode: 2 | Total Reward: -1019.0
--------------------------------------------------
High-Level Goal: REDUCE_CONGESTION
Episode: 3 | Total Reward: -1156.0
--------------------------------------------------
High-Level Goal: REDUCE_WAITING
Episode: 4 | Total Reward: -704.0
--------------------------------------------------
High-Level Goal: REDUCE_CONGESTION
Episode: 5 | Total Reward: -938.0
--------------------------------------------------
High-Level Goal: NORMAL_TRAFFIC
Episode: 6 | Total Reward: -949.0
--------------------------------------------------
High-Level Goal: NORMAL_TRAFFIC
Episode: 7 | Total Reward: -792.0
--------------------------------------------------
High-Level Goal: NORMAL_TRAFFIC
Episode: 8 | Total Reward: -76